# Ambisonic Loudness & Soundscape Meter

Misst für B-Format/Ambisonics-Signale (ACN/SN3D):

1. **Broadcast-Loudness** nach ITU-R BS.1770-5 / EBU R 128:
   Integrated LUFS, Short-Term, Momentary, LRA, True Peak, W-basierte Schätzung
   (AES-Studie *"Loudness Perception of Scene-Based Audio across Loudspeaker
   Configurations and HOA Orders"*) und Richtungs-Loudness pro virtuellem Lautsprecher.
2. **Indoor-Soundscape-Parameter** nach Torresin et al. (2020),
   *Building and Environment* 182, 107152 (aufgenommen mit FOA-Mikrofon):
   L_Aeq, N10/N90 (ISO 532-1), L_A10−L_A90, N10−N90,
   Fluctuation Strength, Roughness, L_C−L_A, Sharpness.

In [1]:
import sys, os
from pathlib import Path
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / "src" / "soundscape_metrics.py").exists():
        sys.path.insert(0, str(base / "src"))
        break
import numpy as np
import matplotlib.pyplot as plt
from ambisonic_loudness import (encode_plane_wave, measure_ambisonics,
                                print_report)
from soundscape_metrics import measure_soundscape, print_soundscape_report

FS = 48000
DUR = 10.0  # Sekunden
t = np.arange(int(FS * DUR)) / FS
rng = np.random.default_rng(42)

## Szenario-Generator (orientiert an den Kategorien der Studie)

Synthetische B-Format-Szenen (1. Ordnung): *Verkehr* (tieffrequent, moduliert),
*Vögel/Garten* (hochfrequente Chirps), *Musik* (breitbandig, dynamisch).

In [2]:
def source_signal(kind):
    if kind == "traffic":
        engine = 0.4 * np.sin(2 * np.pi * 90 * t) * (1 + 0.5 * np.sin(2 * np.pi * 0.7 * t))
        rumble = 0.2 * rng.standard_normal(len(t))
        low = np.convolve(rumble, np.ones(48) / 48, mode="same")
        return 0.6 * engine + 0.4 * low
    if kind == "birds":
        s = np.zeros(len(t))
        for _ in range(60):
            start = rng.integers(0, len(t) - FS // 4)
            n = rng.integers(200, 1200)
            f0 = rng.uniform(2500, 6000)
            chirp = np.sin(2 * np.pi * (f0 * np.arange(n) / FS +
                                        3e6 * (np.arange(n) / FS) ** 2))
            env = np.hanning(n)
            s[start:start + n] += 0.3 * chirp * env
        return s
    if kind == "music":
        notes = [220, 277, 330, 440, 554]
        s = np.zeros(len(t))
        for i in range(40):
            start = i * len(t) // 40
            f = notes[rng.integers(0, len(notes))] * (1 + rng.normal(0, 0.002))
            n = FS // 4
            s[start:start + n] += 0.3 * np.sin(2 * np.pi * f * np.arange(n) / FS)
        drum = 0.5 * np.convolve(rng.standard_normal(len(t)),
                                 np.ones(8) / 8, mode="same") * (
            np.sin(2 * np.pi * 2.0 * t) > 0)
        return s + drum
    raise ValueError(kind)

scenarios = {"traffic": (0.0, 0.0), "birds": (135.0, 30.0), "music": (60.0, 0.0)}

## 1) Broadcast-Loudness (BS.1770-5 / EBU R 128)

In [3]:
order = 1
for name, (az, el) in scenarios.items():
    sig = source_signal(name)
    b = encode_plane_wave(sig, az, el, order)
    r = measure_ambisonics(b, FS)
    print(f"--- {name} (az {az}°, el {el}°) ---")
    print_report(r)
    print()

--- traffic (az 0.0°, el 0.0°) ---
=== Ambisonic Loudness Report (ITU-R BS.1770-5 / EBU R 128) ===
HOA-Ordnung      : 1 (4 Kanäle)
Integrated       : -14.8 LUFS
Max Short-Term   : -14.6 LUFS
Max Momentary    : -12.1 LUFS
Loudness Range   : 0.3 LU
True Peak        : -18.8 dBTP
W-only (0. Ord.) : -16.9 LUFS
Richtungs-Loudness (Virtuelle Lautsprecher):
  az -150.0° / el   0.0° (G=1.00): -32.3 LUFS
  az -120.0° / el   0.0° (G=2.00): -27.7 LUFS
  az  -90.0° / el   0.0° (G=2.00): -25.9 LUFS
  az  -60.0° / el   0.0° (G=2.00): -24.3 LUFS
  az  -30.0° / el   0.0° (G=1.00): -26.4 LUFS
  az    0.0° / el   0.0° (G=1.00): -26.1 LUFS
  az   30.0° / el   0.0° (G=1.00): -26.4 LUFS
  az   60.0° / el   0.0° (G=2.00): -24.3 LUFS
  az   90.0° / el   0.0° (G=2.00): -25.9 LUFS
  az  120.0° / el   0.0° (G=2.00): -27.7 LUFS
  az  150.0° / el   0.0° (G=1.00): -32.3 LUFS



--- birds (az 135.0°, el 30.0°) ---
=== Ambisonic Loudness Report (ITU-R BS.1770-5 / EBU R 128) ===
HOA-Ordnung      : 1 (4 Kanäle)
Integrated       : -22.3 LUFS
Max Short-Term   : -21.8 LUFS
Max Momentary    : -18.6 LUFS
Loudness Range   : 2.9 LU
True Peak        : -16.7 dBTP
W-only (0. Ord.) : -24.2 LUFS
Richtungs-Loudness (Virtuelle Lautsprecher):
  az -150.0° / el   0.0° (G=1.00): -30.1 LUFS
  az -120.0° / el   0.0° (G=2.00): -28.6 LUFS
  az  -90.0° / el   0.0° (G=2.00): -30.1 LUFS
  az  -60.0° / el   0.0° (G=2.00): -31.1 LUFS
  az  -30.0° / el   0.0° (G=1.00): -34.1 LUFS
  az    0.0° / el   0.0° (G=1.00): -33.1 LUFS
  az   30.0° / el   0.0° (G=1.00): -31.6 LUFS
  az   60.0° / el   0.0° (G=2.00): -27.1 LUFS
  az   90.0° / el   0.0° (G=2.00): -26.0 LUFS
  az  120.0° / el   0.0° (G=2.00): -25.4 LUFS
  az  150.0° / el   0.0° (G=1.00): -28.4 LUFS



--- music (az 60.0°, el 0.0°) ---
=== Ambisonic Loudness Report (ITU-R BS.1770-5 / EBU R 128) ===
HOA-Ordnung      : 1 (4 Kanäle)
Integrated       : -10.0 LUFS
Max Short-Term   : -10.0 LUFS
Max Momentary    : -9.4 LUFS
Loudness Range   : 0.1 LU
True Peak        : -9.3 dBTP
W-only (0. Ord.) : -12.2 LUFS
Richtungs-Loudness (Virtuelle Lautsprecher):
  az -150.0° / el   0.0° (G=1.00): -24.9 LUFS
  az -120.0° / el   0.0° (G=2.00): -22.6 LUFS
  az  -90.0° / el   0.0° (G=2.00): -21.9 LUFS
  az  -60.0° / el   0.0° (G=2.00): -20.3 LUFS
  az  -30.0° / el   0.0° (G=1.00): -21.5 LUFS
  az    0.0° / el   0.0° (G=1.00): -19.9 LUFS
  az   30.0° / el   0.0° (G=1.00): -19.0 LUFS
  az   60.0° / el   0.0° (G=2.00): -15.6 LUFS
  az   90.0° / el   0.0° (G=2.00): -16.0 LUFS
  az  120.0° / el   0.0° (G=2.00): -16.9 LUFS
  az  150.0° / el   0.0° (G=1.00): -21.5 LUFS



In [4]:
name = "traffic"
az, el = scenarios[name]
b = encode_plane_wave(source_signal(name), az, el, order)
r = measure_ambisonics(b, FS)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].plot(np.arange(len(r["short_term_series"])) - 2, r["short_term_series"], label="Short-Term (3s)")
ax[0].plot(np.arange(len(r["momentary_series"])) * 0.1, r["momentary_series"], alpha=0.4, label="Momentary (400ms)")
ax[0].axhline(r["integrated_lufs"], color="k", ls="--", label=f"Integrated {r['integrated_lufs']:.1f} LUFS")
ax[0].set(xlabel="Zeit [s]", ylabel="LUFS", title="Loudness-Zeitverlauf: Verkehr")
ax[0].legend(); ax[0].grid(alpha=0.3)
dirs = r["per_speaker_lufs"]
vals = [d["lufs"] for d in dirs]
azs = [d["azimuth"] for d in dirs]
ax[1].bar(np.arange(len(vals)), vals)
ax[1].set(xticks=np.arange(len(vals))[::2],
          xticklabels=[f"{a:.0f}°" for a in azs[::2]],
          ylabel="LUFS", title="Richtungs-Loudness (virtuelle Lautsprecher)")
plt.tight_layout(); plt.show()

## 2) Indoor-Soundscape-Parameter (Torresin et al. 2020)

Die psychoakustischen Größen werden auf den Schalldruck am Hörplatz bezogen
(omni-Komponente W, skaliert mit 2·√π) und über `spl_offset_db` auf dB SPL
kalibriert (dBFS + spl_offset_db = dB SPL, RMS-Konvention).

In [5]:
SPL_OFFSET = 94.0  # Kalibrierung: 0 dBFS RMS = 94 dB SPL
for name, (az, el) in scenarios.items():
    sig = source_signal(name)
    b = encode_plane_wave(sig, az, el, order)
    omni = 2 * np.sqrt(np.pi) * b[0]  # Schalldruck am Zentrum
    r = measure_soundscape(omni, FS, spl_offset_db=SPL_OFFSET)
    print(f"--- {name} ---")
    print_soundscape_report(r)
    print()

[Warning] when computing sharpness from time-varying loudness, a transient effect appears on the first points. To cut it, use 'skip='


--- traffic ---
=== Indoor-Soundscape-Parameter (Torresin et al. 2020) ===
L_Aeq          : 59.0 dB(A)
L_Ceq          : 78.7 dB(C)
L_C - L_A      : 19.7 dB
L_A10 - L_A90  : 7.2 dB  (L_A10=61.6, L_A90=54.4)
N10            : 15.83 sone
N50            : 13.44 sone
N10 - N90      : 5.12 sone  (N90=10.72)
Sharpness S    : 0.90 acum
Fluctuation FS : 0.06 vacil
Roughness R    : 0.06 asper



[Warning] when computing sharpness from time-varying loudness, a transient effect appears on the first points. To cut it, use 'skip='


/home/appuser/.local/lib/python3.12/site-packages/mosqito/sq_metrics/sharpness/sharpness_din/sharpness_din_from_loudness.py:140: RuntimeWarning: invalid value encountered in divide
  S = 0.11 * sum(N_specific * g * z * 0.1, axis=0) / N


--- birds ---
=== Indoor-Soundscape-Parameter (Torresin et al. 2020) ===
L_Aeq          : 61.3 dB(A)
L_Ceq          : 60.0 dB(C)
L_C - L_A      : -1.3 dB
L_A10 - L_A90  : 17.2 dB  (L_A10=67.0, L_A90=49.8)
N10            : 6.72 sone
N50            : 0.75 sone
N10 - N90      : 6.71 sone  (N90=0.01)
Sharpness S    : 1.31 acum
Fluctuation FS : 0.12 vacil
Roughness R    : 3.03 asper



[Warning] when computing sharpness from time-varying loudness, a transient effect appears on the first points. To cut it, use 'skip='


--- music ---
=== Indoor-Soundscape-Parameter (Torresin et al. 2020) ===
L_Aeq          : 78.1 dB(A)
L_Ceq          : 81.7 dB(C)
L_C - L_A      : 3.6 dB
L_A10 - L_A90  : 10.0 dB  (L_A10=80.7, L_A90=70.7)
N10            : 47.89 sone
N50            : 35.11 sone
N10 - N90      : 31.79 sone  (N90=16.11)
Sharpness S    : 1.24 acum
Fluctuation FS : 0.18 vacil
Roughness R    : 0.35 asper



## Interpretation

Nach Torresin et al. (2020) gilt:

- **Comfort** sinkt mit der Lautheit (beste Prädiktoren: N10 / L_Aeq)
- **Content** steigt mit der zeitlichen Variabilität (L_A10−L_A90)
- **Familiarity** sinkt mit der Sharpness S

Beispielhaft zeigt *traffic* hier hohe Lautheit und L_C−L_A (tieffrequent,
uncomfortable), *birds* hohe Sharpness, *music* hohe Variabilität (Content).

## Eigene Datei analysieren

```python
from ambisonic_loudness import measure_file
r = measure_file("mein_hoa_signal.wav")   # B-Format WAV, (order+1)^2 Kanäle
print_report(r)
```